<a href="https://colab.research.google.com/github/Daniel-Duque/pruebas_clase/blob/main/nlp_y_datos_no_estructurados.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip -q install transformers

In [ ]:
#tomado de https://luv-bansal.medium.com/fine-tuning-bert-for-text-classification-in-pytorch-503d97342db2

In [ ]:
import pandas as pd
import numpy as np
import transformers
import torch
import torch.nn as nn
from torch.utils.data import Dataset
from torch.utils.data import DataLoader
import torch.optim as optim
import torch.nn.functional as F
from torchsummary import summary
from tqdm import tqdm

In [ ]:
pd.read_csv('https://github.com/clairett/pytorch-sentiment-classification/raw/master/data/SST2/train.tsv', delimiter='\t', header=None)

,0,1
0,"a stirring , funny and finally transporting re...",1
1,apparently reassembled from the cutting room f...,0
2,they presume their audience wo n't sit still f...,0
3,this is a visually stunning rumination on love...,1
4,jonathan parker 's bartleby should have been t...,1
...,...,...
6915,"painful , horrifying and oppressively tragic ,...",1
6916,take care is nicely performed by a quintet of ...,0
6917,"the script covers huge , heavy topics in a bla...",0
6918,a seriously bad film with seriously warped log...,0


In [ ]:
class BertDataset(Dataset):
    def __init__(self, tokenizer,max_length):
        super(BertDataset, self).__init__()
        self.train_csv=pd.read_csv('https://github.com/clairett/pytorch-sentiment-classification/raw/master/data/SST2/train.tsv', delimiter='\t', header=None)
        self.tokenizer=tokenizer
        self.target=self.train_csv.iloc[:,1]
        self.max_length=max_length

    def __len__(self):
        return len(self.train_csv)

    def __getitem__(self, index):

        text1 = self.train_csv.iloc[index,0]

        inputs = self.tokenizer.encode_plus(
            text1 ,
            None,
            padding='max_length',  # Use padding='max_length' instead of pad_to_max_length=True
            add_special_tokens=True,
            return_attention_mask=True,
            max_length=self.max_length,
            truncation=True,
            return_tensors='pt' # Return PyTorch tensors
        )

        ids = inputs["input_ids"].squeeze() # Remove the batch dimension added by return_tensors='pt'
        token_type_ids = inputs["token_type_ids"].squeeze() # Remove the batch dimension
        mask = inputs["attention_mask"].squeeze() # Remove the batch dimension


        return {
            'ids': ids,
            'mask': mask,
            'token_type_ids': token_type_ids,
            'target': torch.tensor(self.train_csv.iloc[index, 1], dtype=torch.long)
            }
tokenizer = transformers.BertTokenizer.from_pretrained("bert-base-uncased")

dataset= BertDataset(tokenizer, max_length=100)

dataloader=DataLoader(dataset=dataset,batch_size=128)

In [ ]:
class BERT(nn.Module):
    def __init__(self):
        super(BERT, self).__init__()
        self.bert_model = transformers.BertModel.from_pretrained("bert-base-uncased")
        self.medium = nn.Linear(768, 768)
        self.out = nn.Linear(768, 1)

    def forward(self,ids,mask,token_type_ids):
        _,o2= self.bert_model(ids,attention_mask=mask,token_type_ids=token_type_ids, return_dict=False)
        medium=self.medium(o2)
        out= self.out(medium)

        return out

model=BERT()
model

loss_fn = nn.BCEWithLogitsLoss()

#Initialize Optimizer
optimizer= optim.Adam(model.parameters(),lr= 1e-1)


In [ ]:
for param in model.bert_model.parameters():
    param.requires_grad = False

In [ ]:
def finetune(epochs,dataloader,model,loss_fn,optimizer):
    model.train()
    for  epoch in range(epochs):
        print(epoch)

        loop=tqdm(enumerate(dataloader),leave=False,total=len(dataloader))
        for batch, dl in loop:
            ids=dl['ids']
            token_type_ids=dl['token_type_ids']
            mask= dl['mask']
            label=dl['target']
            label = label.unsqueeze(1)

            optimizer.zero_grad()

            output=model(
                ids=ids,
                mask=mask,
                token_type_ids=token_type_ids)
            label = label.type_as(output)

            loss=loss_fn(output,label)
            loss.backward()

            optimizer.step()

            pred = np.where(output >= 0, 1, 0)

            num_correct = sum(1 for a, b in zip(pred, label) if a[0] == b[0])
            num_samples = pred.shape[0]
            accuracy = num_correct/num_samples

            print(f'Got {num_correct} / {num_samples} with accuracy {float(num_correct)/float(num_samples)*100:.2f}')

            # Show progress while training
            loop.set_description(f'Epoch={epoch}/{epochs}')
            loop.set_postfix(loss=loss.item(),acc=accuracy)

    return model

In [ ]:
model=finetune(3, dataloader, model, loss_fn, optimizer)

0


Epoch=0/3:   2%|▏         | 1/55 [00:50<45:14, 50.27s/it, acc=0.422, loss=1.45e+3]

Got 54 / 128 with accuracy 42.19


Epoch=0/3:   4%|▎         | 2/55 [01:39<43:57, 49.77s/it, acc=0.453, loss=392]

Got 58 / 128 with accuracy 45.31


...

Epoch=2/3:  96%|█████████▋| 53/55 [44:07<01:38, 49.49s/it, acc=0.742, loss=0.476]

Got 95 / 128 with accuracy 74.22


Epoch=2/3:  98%|█████████▊| 54/55 [44:57<00:49, 49.63s/it, acc=0.68, loss=0.613]

Got 87 / 128 with accuracy 67.97


Got 6 / 8 with accuracy 75.00
